In [3]:
# Train the models for each bias level
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric

feature_cols = ['Age_bucket_u', 'Credit amount_bucket_u', 'Duration_bucket_u']
target_col= 'Risk'  # the target variable
privileged_groups = [{'Sex_num': 1}]
unprivileged_groups = [{'Sex_num': 0}]

e_ratios = [0.9, 0.8, 0.7, 0.6, 0.5]

Accuracy = np.zeros(len(e_ratios))
F1_score = np.zeros(len(e_ratios))


def _as_scalar(value):
    arr = np.asarray(value)
    if arr.size != 1:
        raise ValueError(f'Expected scalar fairness metric, got shape {arr.shape}: {value}')
    return float(arr.reshape(-1)[0])


def _fairness_metrics(dataset_true, dataset_pred): # helper function
    metric = ClassificationMetric(
        dataset_true,
        dataset_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    dataset_metric = BinaryLabelDatasetMetric(
        dataset_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    # Demographic / statistical parity
    spd = _as_scalar(metric.statistical_parity_difference())
    di = _as_scalar(metric.disparate_impact())
    # Equality of opportunity (TPR difference)
    eod = _as_scalar(metric.equal_opportunity_difference())
    # Average odds difference (avg of FPR and TPR diffs)
    aod = _as_scalar(metric.average_odds_difference())
    # Inequality measure
    theil = _as_scalar(metric.theil_index())
    # Individual fairness based on local label agreement
    consistency = _as_scalar(dataset_metric.consistency())

    return np.array([spd, di, eod, aod, theil, consistency])


def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label='bad')
    return np.array([acc, f1])

def measure_fairness(df, X_test, y_pred_log, y_pred_svm):
    test_index = X_test.index
    test_df = df.loc[test_index, feature_cols + ['Sex', 'Risk']].copy()
    label_map = {'good': 0, 'bad': 1}
    test_df['Risk_num'] = test_df['Risk'].map(label_map)
    sex_map = {'male': 1, 'female': 0}
    test_df['Sex_num'] = test_df['Sex'].map(sex_map)
    if test_df['Sex_num'].isna().any():
        print("WARNING: Some Sex values were not mapped. Check sex_map and df['Sex'].unique().")
        print("Unique Sex values:", test_df['Sex'].unique())

    y_true_num = test_df['Risk_num'].values
    y_pred_log_num = pd.Series(y_pred_log, index=test_index).map(label_map).values
    y_pred_svm_num = pd.Series(y_pred_svm, index=test_index).map(label_map).values
    
    # True dataset
    dataset_true = BinaryLabelDataset(
        favorable_label=0,          # 0 = 'good'
        unfavorable_label=1,        # 1 = 'bad'
        df=test_df[feature_cols + ['Sex_num']].assign(Risk_num=y_true_num),
        label_names=['Risk_num'],
        protected_attribute_names=['Sex_num']
    )

    # Logistic regression predictions
    dataset_pred_log = dataset_true.copy()
    dataset_pred_log.labels = y_pred_log_num.reshape(-1, 1)

    # Linear SVM predictions
    dataset_pred_svm = dataset_true.copy()
    dataset_pred_svm.labels = y_pred_svm_num.reshape(-1, 1)
    
    privileged_groups = [{'Sex_num': 1}]
    unprivileged_groups = [{'Sex_num': 0}]
    log_fairness = _fairness_metrics(dataset_true, dataset_pred_log)
    svm_fairness = _fairness_metrics(dataset_true, dataset_pred_svm)
    return {
        'Logistic Regression': log_fairness,
        'Linear SVM': svm_fairness
    }


def trainmodel(df):
    X = df[feature_cols]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y  # helps preserve class balance
    )

    preprocess = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols)
        ]
    )

    log_reg_clf = Pipeline(steps=[
        ('preprocess', preprocess),
        ('model', LogisticRegression(max_iter=1000, solver='liblinear'))
    ])

    svm_clf = Pipeline(steps=[
        ('preprocess', preprocess),
        ('model', LinearSVC())
    ])

    log_reg_clf.fit(X_train, y_train)
    svm_clf.fit(X_train, y_train)

    y_pred_log = log_reg_clf.predict(X_test)
    y_pred_svm = svm_clf.predict(X_test)

    perf_log = evaluate_model(y_test, y_pred_log)
    perf_svm = evaluate_model(y_test, y_pred_svm)

    f = measure_fairness(df, X_test, y_pred_log, y_pred_svm)
    return {
        'Logistic Regression': np.concatenate([perf_log, f['Logistic Regression']]),
        'Linear SVM': np.concatenate([perf_svm, f['Linear SVM']])
    }




results = {}

for e in e_ratios:
    df = pd.read_csv(f'datasets/german_credit_data_ebiased{e}.csv', delimiter=',')
    results[e] = trainmodel(df)

results



{0.9: {'Logistic Regression': array([ 0.70333333,  0.18348624, -0.0047619 ,  0.99492386,  0.01156987,
         -0.00499093,  0.08651181,  0.97466667]),
  'Linear SVM': array([ 0.70333333,  0.18348624, -0.0047619 ,  0.99492386,  0.01156987,
         -0.00499093,  0.08651181,  0.97466667])},
 0.8: {'Logistic Regression': array([ 0.70333333,  0.22608696, -0.01804453,  0.98042169, -0.01948052,
         -0.01238576,  0.09633626,  0.97333333]),
  'Linear SVM': array([ 0.70333333,  0.22608696, -0.01804453,  0.98042169, -0.01948052,
         -0.01238576,  0.09633626,  0.97333333])},
 0.7: {'Logistic Regression': array([0.70666667, 0.21428571, 0.00172668, 1.00186444, 0.00417493,
         0.00208746, 0.08953367, 0.97666667]),
  'Linear SVM': array([0.70666667, 0.25423729, 0.01604794, 1.01780181, 0.02515395,
         0.01257698, 0.09935395, 0.97666667])},
 0.6: {'Logistic Regression': array([ 0.7       ,  0.26229508, -0.01507953,  0.98320874, -0.02146465,
         -0.01130836,  0.10980195,  0.968

In [4]:
metric_names = ['Accuracy', 'F1', 'SPD', 'DI', 'EOD', 'AOD', 'Theil', 'Consistency']

for model_name in ['Logistic Regression', 'Linear SVM']:
    table = pd.DataFrame(
        {e: results[e][model_name] for e in e_ratios},
        index=metric_names
    )
    print(f"\n{model_name}")
    display(table)



Logistic Regression


,0.9,0.8,0.7,0.6,0.5
Accuracy,0.703333,0.703333,0.706667,0.700000,0.683333
F1,0.183486,0.226087,0.214286,0.262295,0.201681
SPD,-0.004762,-0.018045,0.001727,-0.015080,0.016835
DI,0.994924,0.980422,1.001864,0.983209,1.018746
EOD,0.011570,-0.019481,0.004175,-0.021465,0.051768
AOD,-0.004991,-0.012386,0.002087,-0.011308,-0.006950
Theil,0.086512,0.096336,0.089534,0.109802,0.114436
Consistency,0.974667,0.973333,0.976667,0.968667,0.958000



Linear SVM


,0.9,0.8,0.7,0.6,0.5
Accuracy,0.703333,0.703333,0.706667,0.700000,0.680000
F1,0.183486,0.226087,0.254237,0.274194,0.172414
SPD,-0.004762,-0.018045,0.016048,-0.005371,0.002272
DI,0.994924,0.980422,1.017802,0.993954,1.002490
EOD,0.011570,-0.019481,0.025154,-0.014520,0.044823
AOD,-0.004991,-0.012386,0.012577,0.000228,-0.026552
Theil,0.086512,0.096336,0.099354,0.113120,0.111309
Consistency,0.974667,0.973333,0.976667,0.962000,0.962667
